# Datamine FACTOR Process Example

This notebook demonstrates how to configure and run the **`factor`** process wrapper in `dmstudio`.

## Process Description

## FACTOR

Define a number of factors from the common inter-correlation between variables using the correlation (R) matrix.

R - mode factor analysis using the unrotated, varimax and promax rotated correlation matrix to classify variables into groups (factors) and calculate factor scores. This defines a number of factors from the common inter-correlation between variables using the correlation (R) matrix.

Note the difference from the principal components analysis which uses variance. The number of factors selected is usually less than the number of variables. Three types of factor loadings are calculated, a) the un-rotated solution, ie principal components, b) the rotated varimax solution and c) the oblique promax solution.

Statistical factor analysis is completed by iterative calculation of the un-rotated factor matrix. Non- statistical factor analysis is completed by a single pass only, ie. the varimax and promax rotations are non statistical. The number of factors selected can be forced by defining a given eigen value or a given number of factors. Significance levels are calculated using the Burt-Banks equation.

### FACTOR and PCA Processes

Note the difference from the [principal components analysis](<pca.md>) calculation, which uses variance.

The number of factors selected is usually less than the number of variables. Three types of factor loadings are calculated, a) the un-rotated solution, ie principal components, b) the rotated varimax solution and c) the oblique promax solution. Statistical factor analysis is completed by iterative calculation of the un-rotated factor matrix. Non- statistical factor analysis is completed by a single pass only, ie. the varimax and promax rotations are non statistical. The number of factors selected can be forced by defining a given eigen value or a given number of factors. Significance levels are calculated using the Burt-Banks equation.

### File Handling

The input data file (&**IN**) must have sample identifier field (@**SAMPID**) which has to be declared on input. Optional output files, for un-rotated (&**SCORES**), rotated (**RSCORES**) and oblique rotated (&**OSCORES**) are available for farther processing.

If the user wishes to plot maps of the output scores then the scores files can be joined to the original input file using the **[JOIN](<join.md>)** process and defining **SAMPID** as the keyfield.

**Note** : There is a limit of 45 variables. There is no limit on the number of samples.

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

### Output Files:

* **uscores** (Undefined):
  Optional output file for unrotated factor scores.
  Required=No

* **rscores** (Undefined):
  Optional output file for varimax factor scores.
  Required=No

* **oscores** (Undefined):
  Optional output file for promax factor scores.
  Required=No

### Fields:

* **sampid** (Any : IN):
  Field containing sample identification
  Default=Undefined
  Required=Yes

* **f1** (Numeric : IN):
  First field to be used. No fields specified means all.
  Default=Undefined
  Required=No

* **f2_f10** (Numeric : IN):
  Second and subsequent fields to be used.
  Default=Undefined
  Required=No

### Parameters:

* **maxit**:
  Number of iterations required, valid only for the unrotated PCA option in factor
  analysis. =(0) Non statistical factor analysis. =10 Maximum number of iterations allowed
  for statistical factor analysis.
  Range=0,10
  Values=0,1,2,3,4,5,6,,7,8,9,10
  Default=0
  Required=No

* **eigenmin**:

* **Options** ((1): Eigenvalue required to select the number of components.):
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **numeigen**:

* **Options** ((0): Maximum number of eigenvalues is set to the number of fields or to 10,):
  whichever is the lower.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **promaxcf**:

* **Options** ((3): Promax oblique rotation exponent. Range is 1-9.):
  Range=1,9
  Values=Undefined
  Default=3
  Required=No

* **print**:
  > 0 Display scores on the screen (0). Note - Do not use for large files.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('factor')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute factor
print("Running factor...")
dm_cmd.factor(
    in_i='t_assays',  # required
    uscores_o=['t_factor_out'],  # required
    rscores_o=['t_factor_out'],  # required
    oscores_o=['t_factor_out'],  # required
    sampid_f='optional',  # required
    # f1_f='optional',  # optional
    # f2_f10_f='optional',  # optional
    # maxit_p=0,  # optional
    # eigenmin_p=1,  # optional
    # numeigen_p=0,  # optional
    # promaxcf_p=3,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("factor execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_factor_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")